# Neervalam AI Model Training & Benchmarking Notebook
**Domain:** Groundwater Level Forecasting & Regional Drought Early Warning  
**Dataset:** Neervalam Audited CGWB Dataset (2,524 master observations, 1,980 transition rows)  
**Target:** Predict next-season water level ($m$ bgl) and classify Drought Severity (`Normal`, `Watch`, `Warning`, `Emergency`).


In [ ]:
import os, sys, json
import numpy as np, pandas as pd
import joblib
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score, classification_report
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")


## 1. Load Clean Modelling Tables
Loads `stage1_modelling_table_ALL_YEARS.csv` refreshed across all continuous CGWB Year Books (2019 to 2025).


In [ ]:
DATA_PATH = os.path.abspath("../../../../data/neervalam_dataset/06_model_features/stage1_modelling_table_ALL_YEARS.csv")
df = pd.read_csv(DATA_PATH)
print("Dataset Shape:", df.shape)
print("Unique Wells:", df['well_id'].nunique())
print("Target Months:", sorted(df['target_month'].unique()))
df.head(3)


## 2. Grouped Well Holdout Split
Splits by **Well ID** to strictly prevent spatial and temporal data leakage between training and testing sets.


In [ ]:
unique_wells = df['well_id'].unique()
np.random.seed(42)
shuffled = np.random.permutation(unique_wells)
n_train, n_val = int(len(shuffled)*0.70), int(len(shuffled)*0.15)

train_wells = set(shuffled[:n_train])
val_wells = set(shuffled[n_train:n_train+n_val])
test_wells = set(shuffled[n_train+n_val:])

train_df = df[df['well_id'].isin(train_wells)]
val_df = df[df['well_id'].isin(val_wells)]
test_df = df[df['well_id'].isin(test_wells)]

print(f"Train: {len(train_wells)} wells ({len(train_df)} rows)")
print(f"Val:   {len(val_wells)} wells ({len(val_df)} rows)")
print(f"Test:  {len(test_wells)} wells ({len(test_df)} rows)")


## 3. Train Candidate Models & Benchmark vs Persistence
We compare:
1. **Naive Persistence Baseline** (Next Level = Current Level, benchmark error **2.13m MAE**)
2. **Ridge Regression**
3. **Random Forest Regressor**
4. **LightGBM Regressor**


In [ ]:
feature_cols = ['wl_t_mbgl', 'month_t', 'month_sin_t', 'month_cos_t', 'water_year_t', 'gap_days', 'hist_n_obs', 'hist_mean_mbgl', 'hist_min_mbgl', 'hist_max_mbgl', 'wl_t_minus_hist_mean_m', 'transmissivity_m2_day', 'storativity_s', 'weathered_zone_thickness_m', 'specific_yield_pct', 'rain_7d_mm', 'rain_30d_mm', 'rain_90d_mm', 'et0_30d_mm', 'temp_mean_30d_c', 'humidity_mean_30d_pct', 'well_type_code', 'block_code']
target_col = 'target_wl_next_mbgl'

X_train, y_train = train_df[feature_cols].fillna(0), train_df[target_col]
X_test, y_test = test_df[feature_cols].fillna(0), test_df[target_col]

# Models
pers_pred = test_df['wl_t_mbgl'].values
ridge = Ridge(alpha=10.0, random_state=42).fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1).fit(X_train, y_train)
lgb_model = lgb.LGBMRegressor(n_estimators=180, learning_rate=0.04, max_depth=8, random_state=42, verbose=-1).fit(X_train, y_train)

models = {
    "Persistence Baseline": pers_pred,
    "Ridge Regression": ridge.predict(X_test),
    "Random Forest Regressor": rf.predict(X_test),
    "LightGBM Regressor": lgb_model.predict(X_test)
}

metrics = []
for name, pred in models.items():
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    metrics.append({"Model": name, "MAE (m)": round(mae, 3), "RMSE (m)": round(rmse, 3), "R2 Score": round(r2, 3)})

pd.DataFrame(metrics).sort_values("MAE (m)")


## 4. Government Drought Severity Alert Classifier
Classifies block drought risk into **Normal (0)**, **Watch (1)**, **Warning (2)**, and **Emergency (3)**.


In [ ]:
def get_drought_label(row):
    anomaly = row.get("wl_t_minus_hist_mean_m", 0)
    rain_90d = row.get("rain_90d_mm", 100)
    if anomaly > 4.0 and rain_90d < 60:
        return 3 # Emergency
    elif anomaly > 2.0 or (anomaly > 1.0 and rain_90d < 100):
        return 2 # Warning
    elif anomaly > 0.5 or rain_90d < 150:
        return 1 # Watch
    return 0 # Normal

df['drought_label'] = df.apply(get_drought_label, axis=1)
d_feats = [c for c in ['wl_t_mbgl', 'wl_t_minus_hist_mean_m', 'month_t', 'rain_30d_mm', 'rain_90d_mm', 'et0_30d_mm'] if c in df.columns]

clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1)
clf.fit(df[d_feats].fillna(0), df['drought_label'])
preds = clf.predict(df[d_feats].fillna(0))

print("Accuracy:", round(accuracy_score(df['drought_label'], preds)*100, 2), "%")
print("Macro F1:", round(f1_score(df['drought_label'], preds, average='macro'), 4))
print(classification_report(df['drought_label'], preds, target_names=['Normal', 'Watch', 'Warning', 'Emergency'], digits=3))


## 5. Model Serialization & Export
Saves the trained models for backend inference via FastAPI.


In [ ]:
SAVED_DIR = os.path.abspath("../saved_models")
joblib.dump(rf, os.path.join(SAVED_DIR, "water_level_rf_model.joblib"))
joblib.dump(lgb_model, os.path.join(SAVED_DIR, "water_level_lgbm_model.joblib"))
joblib.dump(clf, os.path.join(SAVED_DIR, "drought_classifier_model.joblib"))
print("Saved all models to:", SAVED_DIR)
